Modified RelBench code. 
Original code is available at [link](https://github.com/snap-stanford/relbench/blob/main/tutorials/train_model.ipynb)

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [187]:
import pandas as pd
import torch.nn as nn
from relbench.datasets import Dataset, get_dataset
from relbench.tasks import BaseTask, get_task

from predql import TConverter
from predql.base import Table

In [188]:
def get_timestamps(dataset             : Dataset,
                   timedelta           : pd.Timedelta,
                   num_eval_timestamps : int,
                   split               : str) -> "pd.Series[pd.Timestamp]":
    db = dataset.get_db(upto_test_timestamp=(split != "test"))

    corrective_timedelta = pd.Timedelta("1ms")

    if split == "train":
        start = dataset.val_timestamp - timedelta + corrective_timedelta
        end = db.min_timestamp + corrective_timedelta
        freq = -timedelta
    elif split == "val":
        start = dataset.val_timestamp + corrective_timedelta
        end = min(
            dataset.val_timestamp
            + timedelta * (num_eval_timestamps - 1),
            dataset.test_timestamp - timedelta,
            ) + corrective_timedelta
        freq = timedelta
    elif split == "test":
        start = dataset.test_timestamp + corrective_timedelta
        end = min(
            dataset.test_timestamp
            + timedelta * (num_eval_timestamps - 1),
            db.max_timestamp - timedelta,
            ) + corrective_timedelta
        freq = timedelta
    else:
        pass

    timestamps = pd.date_range(start=start, end=end, freq=freq)
    return timestamps

In [189]:
def get_table(dataset      : Dataset,
              task         : BaseTask,
              split        : str,
              predql_query : str) -> Table:
    timestamps = get_timestamps(
        dataset=dataset,
        timedelta=task.timedelta,
        num_eval_timestamps=task.num_eval_timestamps,
        split=split,
    )

    converter = TConverter(dataset.get_db(upto_test_timestamp=(split != "test")), timestamps)

    return converter.convert(predql_query)

In [190]:
def rename_table_df(table         : Table,
                    new_fk        : str,
                    new_timestamp : str,
                    new_label     : str) -> Table:
    new_df = table.df.rename(columns={'fk'        : new_fk,
                                      'timestamp' : new_timestamp,
                                      'label'     : new_label})
    new_table = Table(df=new_df,
                      fkey_col_to_pkey_table={new_fk : table.fkey_col_to_pkey_table.get('fk')},
                      pkey_col=table.pkey_col,
                      time_col=new_timestamp)
    return new_table

In [191]:
dataset = get_dataset("rel-f1", download=True)
task = get_task("rel-f1", "driver-position", download=False)

out_channels = 1
loss_fn = nn.L1Loss()
tune_metric = "mae"
higher_is_better = False

In [192]:
predql_query = """
    PREDICT AVG(results.positionOrder, 0, 60, DAYS)
    FOR EACH drivers.driverId
    WHERE AVG(results.positionOrder, 0, 60, DAYS) IS NOT NULL;
"""

In [202]:
# PredQL tables

train_table = rename_table_df(get_table(dataset, task, "train", predql_query),
                              new_fk="driverId",
                              new_timestamp="date",
                              new_label="position")
val_table = rename_table_df(get_table(dataset, task, "val", predql_query),
                              new_fk="driverId",
                              new_timestamp="date",
                              new_label="position")
test_table = rename_table_df(get_table(dataset, task, "test", predql_query),
                              new_fk="driverId",
                              new_timestamp="date",
                              new_label="position")

In [194]:
# RelBench tables

train_table = task.get_table("train")
val_table = task.get_table("val")
test_table = task.get_table("test")

In [203]:
print(train_table)

------------------ Table ------------------
DataFrame:
      driverId                    date  position
0          498 1950-06-19 00:00:00.001      18.0
1          579 1950-06-19 00:00:00.001       1.0
2          589 1950-06-19 00:00:00.001      15.0
3          626 1950-06-19 00:00:00.001       4.0
4          627 1950-06-19 00:00:00.001       9.0
...        ...                     ...       ...
7448        41 2004-09-03 00:00:00.001       7.0
7449        43 2004-09-03 00:00:00.001      16.0
7450        44 2004-09-03 00:00:00.001      17.0
7451        45 2004-09-03 00:00:00.001      17.0
7452        46 2004-09-03 00:00:00.001      16.0

[7453 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'driverId': 'drivers'}
Primary Key Column: None
Time Column: date
-------------------------------------------


In [204]:
print(val_table)

------------------ Table ------------------
DataFrame:
     driverId                    date   position
0           1 2005-03-02 00:00:00.001  11.000000
1           3 2005-03-02 00:00:00.001   1.500000
2           7 2005-03-02 00:00:00.001   9.000000
3          10 2005-03-02 00:00:00.001  16.333333
4          12 2005-03-02 00:00:00.001   9.250000
..        ...                     ...        ...
494        23 2009-10-07 00:00:00.001  13.500000
495        66 2009-10-07 00:00:00.001   7.500000
496       152 2009-10-07 00:00:00.001  17.000000
497       153 2009-10-07 00:00:00.001  15.500000
498       154 2009-10-07 00:00:00.001   8.000000

[499 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'driverId': 'drivers'}
Primary Key Column: None
Time Column: date
-------------------------------------------


In [205]:
print(test_table)

------------------ Table ------------------
DataFrame:
     driverId                    date   position
0           0 2010-03-02 00:00:00.001   4.250000
1           2 2010-03-02 00:00:00.001   4.000000
2           3 2010-03-02 00:00:00.001   5.500000
3           4 2010-03-02 00:00:00.001  15.000000
4           8 2010-03-02 00:00:00.001   5.500000
..        ...                     ...        ...
755       830 2016-05-29 00:00:00.001  15.333333
756       831 2016-05-29 00:00:00.001  10.333333
757       834 2016-05-29 00:00:00.001  16.666667
758       835 2016-05-29 00:00:00.001  17.000000
759       836 2016-05-29 00:00:00.001  18.000000

[760 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'driverId': 'drivers'}
Primary Key Column: None
Time Column: date
-------------------------------------------


In [206]:
import math
import os

import numpy as np
import torch
from torch_geometric.seed import seed_everything

seed_everything(42)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
root_dir = "./data"

cuda


The first big move is to build a graph out of the database. Here we use our pre-prepared conversion function.

The source code can be found at: https://github.com/snap-stanford/relbench/blob/main/relbench/modeling/graph.py

Each node in the graph corresonds to a single row in the database. Crucially, PyTorch Frame stores whole tables as objects in a way that is compatibile with PyG minibatch sampling, meaning we can sample subgraphs as in https://arxiv.org/abs/1706.02216, and retrieve the relevant raw features.

PyTorch Frame also stores the `stype` (i.e., modality) of each column, and any specialized feature encoders (e.g., text encoders) to be used later. So we need to configure the `stype` for each column, for which we use a function that tries to automatically detect the `stype`.

In [207]:
from relbench.modeling.utils import get_stype_proposal

db = dataset.get_db()
col_to_stype_dict = get_stype_proposal(db)
col_to_stype_dict

{'drivers': {'driverId': <stype.numerical: 'numerical'>,
  'driverRef': <stype.text_embedded: 'text_embedded'>,
  'code': <stype.text_embedded: 'text_embedded'>,
  'forename': <stype.text_embedded: 'text_embedded'>,
  'surname': <stype.text_embedded: 'text_embedded'>,
  'dob': <stype.timestamp: 'timestamp'>,
  'nationality': <stype.text_embedded: 'text_embedded'>},
 'races': {'raceId': <stype.numerical: 'numerical'>,
  'year': <stype.categorical: 'categorical'>,
  'round': <stype.numerical: 'numerical'>,
  'circuitId': <stype.numerical: 'numerical'>,
  'name': <stype.text_embedded: 'text_embedded'>,
  'date': <stype.timestamp: 'timestamp'>,
  'time': <stype.timestamp: 'timestamp'>},
 'constructor_standings': {'constructorStandingsId': <stype.numerical: 'numerical'>,
  'raceId': <stype.numerical: 'numerical'>,
  'constructorId': <stype.numerical: 'numerical'>,
  'points': <stype.numerical: 'numerical'>,
  'position': <stype.numerical: 'numerical'>,
  'wins': <stype.numerical: 'numerical

If trying a new dataset, you should definitely check through this dict of `stype`s to check that look right, and manually change any mistakes by the auto-detection function.

Next we also define our text encoding model, which we use GloVe embeddings for speed and convenience. Feel free to try alternatives here.

In [208]:
from typing import List, Optional

from sentence_transformers import SentenceTransformer


class GloveTextEmbedding:
    def __init__(self, device: Optional[torch.device] = None):
        self.model = SentenceTransformer(
            "sentence-transformers/average_word_embeddings_glove.6B.300d",
            device=device,
        )

    def __call__(self, sentences: List[str]) -> torch.Tensor:
        return torch.from_numpy(self.model.encode(sentences))

In [209]:
from getpass import getpass

import relbench.modeling.graph
import relbench.modeling.utils
from relbench.modeling.graph import make_pkey_fkey_graph
from torch_frame.config.text_embedder import TextEmbedderConfig

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Insert HF token: ")

text_embedder_cfg = TextEmbedderConfig(
    text_embedder=GloveTextEmbedding(device=device), batch_size=256
)

def patched_to_unix_time(ser):
    unix_time = ser.astype("int64").values.copy()
    if ser.dtype == np.dtype("datetime64[ns]"):
        unix_time //= 10**9
    return unix_time

relbench.modeling.graph.to_unix_time = patched_to_unix_time
relbench.modeling.utils.to_unix_time = patched_to_unix_time

data, col_stats_dict = make_pkey_fkey_graph(
    db,
    col_to_stype_dict=col_to_stype_dict,  # speficied column types
    text_embedder_cfg=text_embedder_cfg,  # our chosen text encoder
    cache_dir=os.path.join(
        root_dir, "rel-f1_materialized_cache"
    ),  # store materialized graph for convenience
)

/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globa

We can now check out `data`, our main graph object. `data` is a heterogeneous and temporal graph, with node types given by the table it originates from.

In [210]:
data

HeteroData(
  drivers={ tf=TensorFrame([857, 6]) },
  races={
    tf=TensorFrame([820, 5]),
    time=[820],
  },
  constructor_standings={
    tf=TensorFrame([10170, 4]),
    time=[10170],
  },
  standings={
    tf=TensorFrame([28115, 4]),
    time=[28115],
  },
  qualifying={
    tf=TensorFrame([4082, 3]),
    time=[4082],
  },
  circuits={ tf=TensorFrame([77, 7]) },
  constructor_results={
    tf=TensorFrame([9408, 2]),
    time=[9408],
  },
  results={
    tf=TensorFrame([20323, 11]),
    time=[20323],
  },
  constructors={ tf=TensorFrame([211, 3]) },
  (races, f2p_circuitId, circuits)={ edge_index=[2, 820] },
  (circuits, rev_f2p_circuitId, races)={ edge_index=[2, 820] },
  (constructor_standings, f2p_raceId, races)={ edge_index=[2, 10170] },
  (races, rev_f2p_raceId, constructor_standings)={ edge_index=[2, 10170] },
  (constructor_standings, f2p_constructorId, constructors)={ edge_index=[2, 10170] },
  (constructors, rev_f2p_constructorId, constructor_standings)={ edge_index=[2, 1

We can also check out the TensorFrame for one table like this:

In [211]:
data["races"].tf

TensorFrame(
  num_cols=5,
  num_rows=820,
  categorical (1): ['year'],
  numerical (1): ['round'],
  timestamp (2): ['date', 'time'],
  embedding (1): ['name'],
  has_target=False,
  device='cpu',
)

This may be a little confusing at first, as in graph ML it is more standard to associate to the graph object `data` a tensor, e.g., `data.x` for which `data.x[idx]` is a 1D array/tensor storing all the features for node with index `idx`.

But actually this `data` object behaves similarly. For a given node type, e.g., `races` again, `data['races']` stores two pieces of information


In [212]:
list(data["races"].keys())

['tf', 'time']

A `TensorFrame` object, and a timestamp for each node. The `TensorFrame` object acts analogously to the usual tensor of node features, and you can simply use indexing to retrieve the features of a single row (node), or group of nodes.

In [213]:
data["races"].tf[10]

TensorFrame(
  num_cols=5,
  num_rows=1,
  categorical (1): ['year'],
  numerical (1): ['round'],
  timestamp (2): ['date', 'time'],
  embedding (1): ['name'],
  has_target=False,
  device='cpu',
)

In [214]:
data["races"].tf[10:20]

TensorFrame(
  num_cols=5,
  num_rows=10,
  categorical (1): ['year'],
  numerical (1): ['round'],
  timestamp (2): ['date', 'time'],
  embedding (1): ['name'],
  has_target=False,
  device='cpu',
)

We can also check the edge indices between two different node types, such as `races` amd `circuits`. Note that the edges are also heterogenous, so we also need to specify which edge type we want to look at. Here we look at `f2p_curcuitId`, which are the directed edges pointing _from_ a race (the `f` stands for `foreign key`), _to_ the circuit at which te race happened (the `p` stands for `primary key`).

In [215]:
data[("races", "f2p_circuitId", "circuits")]

{'edge_index': tensor([[  0,   1,   2,  ..., 817, 818, 819],
        [  8,   5,  18,  ...,  21,  17,  23]])}

Now we are ready to instantiate our data loaders. For this we will need to import PyTorch Geometric, our GNN library. Whilst we're at it let's add a seed.


In [216]:
from relbench.modeling.graph import get_node_train_table_input, make_pkey_fkey_graph
from torch_geometric.loader import NeighborLoader

loader_dict = {}

for split, table in [
    ("train", train_table),
    ("val", val_table),
    ("test", test_table),
]:
    table_input = get_node_train_table_input(
        table=table,
        task=task,
    )
    entity_table = table_input.nodes[0]
    loader_dict[split] = NeighborLoader(
        data,
        num_neighbors=[
            128 for i in range(2)
        ],  # we sample subgraphs of depth 2, 128 neighbors per node.
        time_attr="time",
        input_nodes=table_input.nodes,
        input_time=table_input.time,
        transform=table_input.transform,
        batch_size=512,
        temporal_strategy="uniform",
        shuffle=split == "train",
        num_workers=0,
        persistent_workers=False,
    )

Now we need our model...




In [217]:
import copy
from typing import Any, Dict, List

import torch
from relbench.modeling.nn import HeteroEncoder, HeteroGraphSAGE, HeteroTemporalEncoder
from torch import Tensor
from torch.nn import Embedding, ModuleDict
from torch_frame.data.stats import StatType
from torch_geometric.data import HeteroData
from torch_geometric.nn import MLP
from torch_geometric.typing import NodeType


class Model(torch.nn.Module):

    def __init__(
        self,
        data: HeteroData,
        col_stats_dict: Dict[str, Dict[str, Dict[StatType, Any]]],
        num_layers: int,
        channels: int,
        out_channels: int,
        aggr: str,
        norm: str,
        # List of node types to add shallow embeddings to input
        shallow_list: List[NodeType] = [],
        # ID awareness
        id_awareness: bool = False,
    ):
        super().__init__()

        self.encoder = HeteroEncoder(
            channels=channels,
            node_to_col_names_dict={
                node_type: data[node_type].tf.col_names_dict
                for node_type in data.node_types
            },
            node_to_col_stats=col_stats_dict,
        )
        self.temporal_encoder = HeteroTemporalEncoder(
            node_types=[
                node_type for node_type in data.node_types if "time" in data[node_type]
            ],
            channels=channels,
        )
        self.gnn = HeteroGraphSAGE(
            node_types=data.node_types,
            edge_types=data.edge_types,
            channels=channels,
            aggr=aggr,
            num_layers=num_layers,
        )
        self.head = MLP(
            channels,
            out_channels=out_channels,
            norm=norm,
            num_layers=1,
        )
        self.embedding_dict = ModuleDict(
            {
                node: Embedding(data.num_nodes_dict[node], channels)
                for node in shallow_list
            }
        )

        self.id_awareness_emb = None
        if id_awareness:
            self.id_awareness_emb = torch.nn.Embedding(1, channels)
        self.reset_parameters()

    def reset_parameters(self):
        self.encoder.reset_parameters()
        self.temporal_encoder.reset_parameters()
        self.gnn.reset_parameters()
        self.head.reset_parameters()
        for embedding in self.embedding_dict.values():
            torch.nn.init.normal_(embedding.weight, std=0.1)
        if self.id_awareness_emb is not None:
            self.id_awareness_emb.reset_parameters()

    def forward(
        self,
        batch: HeteroData,
        entity_table: NodeType,
    ) -> Tensor:
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
            batch.num_sampled_nodes_dict,
            batch.num_sampled_edges_dict,
        )

        return self.head(x_dict[entity_table][: seed_time.size(0)])

    def forward_dst_readout(
        self,
        batch: HeteroData,
        entity_table: NodeType,
        dst_table: NodeType,
    ) -> Tensor:
        if self.id_awareness_emb is None:
            raise RuntimeError(
                "id_awareness must be set True to use forward_dst_readout"
            )
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)
        # Add ID-awareness to the root node
        x_dict[entity_table][: seed_time.size(0)] += self.id_awareness_emb.weight

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
        )

        return self.head(x_dict[dst_table])


model = Model(
    data=data,
    col_stats_dict=col_stats_dict,
    num_layers=2,
    channels=128,
    out_channels=1,
    aggr="sum",
    norm="batch_norm",
).to(device)


# if you try out different RelBench tasks you will need to change these
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
epochs = 20

We also need standard train/test loops

In [218]:
from tqdm import tqdm


def train() -> float:
    model.train()

    loss_accum = count_accum = 0
    for batch in tqdm(loader_dict["train"]):
        batch = batch.to(device)

        optimizer.zero_grad()
        pred = model(
            batch,
            task.entity_table,
        )
        pred = pred.view(-1) if pred.size(1) == 1 else pred

        loss = loss_fn(pred.float(), batch[entity_table].y.float())
        loss.backward()
        optimizer.step()

        loss_accum += loss.detach().item() * pred.size(0)
        count_accum += pred.size(0)

    return loss_accum / count_accum


@torch.no_grad()
def test(loader: NeighborLoader) -> np.ndarray:
    model.eval()

    pred_list = []
    for batch in loader:
        batch = batch.to(device)
        pred = model(
            batch,
            task.entity_table,
        )
        pred = pred.view(-1) if pred.size(1) == 1 else pred
        pred_list.append(pred.detach().cpu())
    return torch.cat(pred_list, dim=0).numpy()

Now we are ready to train!

In [ ]:
state_dict = None
best_val_metric = -math.inf if higher_is_better else math.inf
for epoch in range(1, epochs + 1):
    train_loss = train()
    val_pred = test(loader_dict["val"])
    val_metrics = task.evaluate(val_pred, val_table)
    print(f"Epoch: {epoch:02d}, Train loss: {train_loss}, Val metrics: {val_metrics}")

    if (higher_is_better and val_metrics[tune_metric] > best_val_metric) or (
        not higher_is_better and val_metrics[tune_metric] < best_val_metric
    ):
        best_val_metric = val_metrics[tune_metric]
        state_dict = copy.deepcopy(model.state_dict())


model.load_state_dict(state_dict)
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, val_table)
print(f"Best Val metrics: {val_metrics}")

test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred, test_table)
print(f"Best test metrics: {test_metrics}")

100%|██████████| 15/15 [00:02<00:00,  5.51it/s]


Epoch: 01, Train loss: 9.26776910490371, Val metrics: {'r2': -0.2881502693246545, 'mae': 4.425393091302437, 'rmse': 5.26179051751788}


100%|██████████| 15/15 [00:02<00:00,  5.61it/s]


Epoch: 02, Train loss: 5.9398350887741325, Val metrics: {'r2': -0.37152140482455476, 'mae': 4.335191816382195, 'rmse': 5.429396824137778}


100%|██████████| 15/15 [00:02<00:00,  5.61it/s]


...

Epoch: 18, Train loss: 4.383305073296258, Val metrics: {'r2': 0.29409059330052456, 'mae': 3.109855283381705, 'rmse': 3.895155268883386}


100%|██████████| 15/15 [00:02<00:00,  5.63it/s]


Epoch: 19, Train loss: 4.378604266526122, Val metrics: {'r2': 0.24943839593245665, 'mae': 3.212502590018905, 'rmse': 4.016460141812404}


100%|██████████| 15/15 [00:02<00:00,  5.60it/s]


Epoch: 20, Train loss: 4.351278431924513, Val metrics: {'r2': 0.2538359185326531, 'mae': 3.171877511533484, 'rmse': 4.004676684450457}
Best Val metrics: {'r2': 0.31379003155868934, 'mae': 3.0725076965913027, 'rmse': 3.8404206918593937}
Best test metrics: {'r2': 0.14483699716961196, 'mae': 3.9686224821157627, 'rmse': 4.818305223423789}


In [223]:
# Empty cache if needed

import gc

import torch

gc.collect()

torch.cuda.empty_cache()